# Scrape Tabel BPS

In [13]:
import os, requests, pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("BPS_API_KEY")
assert API_KEY, "API key tidak ditemukan! Cek file .env kamu."

BASE = "https://webapi.bps.go.id/v1/api/list"

def bps_get(params):
    resp = requests.get(BASE, params={**params, "key": API_KEY}, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    assert data["status"] == "OK", f"API error: {data}"
    return data

### Cari var_id

In [14]:
KEYWORD = "kendaraan"

resp  = bps_get({"model": "var", "lang": "ind", "domain": "0000", "page": 1})
pages = resp["data"][0]["pages"]

semua_var = []
for page in range(1, pages + 1):
    r = bps_get({"model": "var", "lang": "ind", "domain": "0000", "page": page})
    semua_var.extend(r["data"][1])

df_var = pd.DataFrame(semua_var)
df_var[df_var["title"].str.lower().str.contains(KEYWORD)][["var_id", "title", "unit"]]

,var_id,title,unit
390,673,Produksi Kendaraan Bermotor dalam Negeri,Unit
564,57,Perkembangan Jumlah Kendaraan Bermotor Menurut...,Unit
1670,2013,Persentase penduduk berumur 10 tahun ke atas y...,Tidak Ada Satuan


### Pilih var_id & cek tahun tersedia

In [15]:
VAR_ID = "57"

resp_th = bps_get({"model": "th", "lang": "ind", "domain": "0000", "var": VAR_ID})
pd.DataFrame(resp_th["data"][1])

,th_id,th
0,124,2024
1,123,2023
2,122,2022
3,121,2021
4,120,2020
5,119,2019
6,118,2018
7,117,2017
8,116,2016
9,115,2015


### Ambil & tampilkan data

In [16]:
TH_ID = "124"

resp_data = bps_get({"model": "data", "lang": "ind", "domain": "0000", "var": VAR_ID, "th": TH_ID})

map_vervar  = {str(v["val"]): v["label"] for v in resp_data["vervar"]}
map_turvar  = {str(v["val"]): v["label"] for v in resp_data["turvar"]}
map_tahun   = {str(v["val"]): str(v["label"]) for v in resp_data["tahun"]}
map_turtahun = {str(v["val"]): v["label"] for v in resp_data["turtahun"]}
var_id_str  = str(resp_data["var"][0]["val"])

rows = []
for ver_id, ver_label in map_vervar.items():
    for tur_id, tur_label in map_turvar.items():
        for th_id, th_label in map_tahun.items():
            for turth_id in map_turtahun.keys():
                key   = f"{ver_id}{var_id_str}{tur_id}{th_id}{turth_id}"
                nilai = resp_data["datacontent"].get(key)
                if nilai is not None:
                    rows.append({
                        resp_data["labelvervar"]: ver_label,
                        "jenis"                 : tur_label,
                        "tahun"                 : th_label,
                        "nilai"                 : nilai,
                    })

df = pd.DataFrame(rows)
df

,Jenis Kendaraan Bermotor,jenis,tahun,nilai
0,Mobil Penumpang,Tidak ada,2024,20444507
1,Mobil Bis,Tidak ada,2024,293991
2,Mobil Barang,Tidak ada,2024,6277403
3,Sepeda motor,Tidak ada,2024,139450013
4,Jumlah,Tidak ada,2024,166465914


In [17]:
# simpan ke CSV
fname = f"bps_var{VAR_ID}_th{TH_ID}.csv"
df.to_csv(fname, index=False)
print(f"Tersimpan → {fname} | {df.shape[0]} baris, {df.shape[1]} kolom")

Tersimpan → bps_var57_th124.csv | 5 baris, 4 kolom
